In [ ]:
import sys
sys.path.append(r'..')
import os
import numpy as np
import json
from matplotlib import pyplot as plt
import matplotlib as mpl
from matplotlib import pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import math
import statistics
import pickle as pkl
import torch

from expression_tree import simplify_equation, calc_r_squared, NMSE_reward_func, BIC_np_calc_loss
from scipy import optimize
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def BIC_np_calc_loss(pred_y, y, normalizer, v, var_count, node_count, bic_scaler=1.0):
    # invert noise
    sample_size = len(y)
    return bic_scaler * (var_count + node_count) * math.log(sample_size) + sample_size * math.log(v)
    
def smooth_data(data, window_size=5):
    # return [max(data[i:i+window_size]) for i in range(len(data)-window_size+1)]
    return [sum(data[i:i+window_size])/window_size for i in range(len(data)-window_size+1)]

In [ ]:
# Loading the dictionary
dataset = "feynman_I_47_23"
seed = 43532

with open(os.getcwd() + f"\\{dataset}_{seed}_0.pkl", 'rb') as file:
        results = pkl.load(file)

In [ ]:
for key in results["Hyper-parameters"].keys():
    print(f"{key}: {results['Hyper-parameters'][key]}")

In [ ]:
for key in results['Training Cycle'][0]['parameters'].keys():
    print(f"{key}: {results['Training Cycle'][0]['parameters'][key]}")

In [ ]:
for key in results['Best Function'].keys(): 
    print(f'{key}: {results["Best Function"][key]}')
time_keys = ['Sample Time', 'Opt Time', 'Reward', 'Prediction']
print(f"Run Time: {round(sum([sum(results['Training Cycle'][0]['Timings'][key]) for key in time_keys])/3600, 2)} hours")
id_epoch = np.argmax(np.array([max(r) for r in results['Training Cycle'][0]["Iteration Info"]["Rewards"]]))
print(f"Discovered Epoch: {id_epoch}")
equ = results["Best Function"]["Equation"]
if len(results["Best Function"]["Constants"]) != 0:
    if isinstance(results["Best Function"]["Constants"][0], float):
        results["Best Function"]["Constants"] = [[r] for r in results["Best Function"]["Constants"]]
    for i, c in enumerate(results["Best Function"]["Constants"]):
        equ = equ.replace(f"c[{i}]", str(round(c[0], 3)))
equ = equ.replace("np.", "torch.")
simplified_equation = simplify_equation(equ, 0, 25)
print(f"Final Equation: {simplified_equation}")


In [ ]:
device = torch.device("cpu")
x = np.array(results['Training Cycle'][0]["Training Data"][0])
c = np.array(results["Best Function"]["Constants"])
y_pred = torch.tensor(eval(results["Best Function"]["Equation"].replace("torch.", "np.")))
y_real = torch.tensor(results['Training Cycle'][0]["Training Data"][1])
v = torch.mean((y_pred - y_real) ** 2)
sample_size = len(y_real)
MSE = torch.mean((y_pred - y_real)**2)
loss = -BIC_np_calc_loss(y_pred, y_real, torch.std(y_real), v, len(c), 16)
r_squared = 1 - torch.sum((y_pred - y_real)**2)/torch.sum((y_real - torch.mean(y_real))**2)
reward = NMSE_reward_func(y_pred.numpy(), y_real.numpy(), y_real.std())

print("Training Data")
print(f"R2: {r_squared}")
print(f"Reward: {reward}")
print(f"MSE: {MSE}")
print(f"NMSE: {MSE/(y_real**2).mean()}")
print(f"Loss: {loss}")

In [ ]:
test_x = results["Test Data"][0]
x = test_x
c = np.array(results["Best Function"]["Constants"])
y_pred = torch.tensor(eval(results["Best Function"]["Equation"]))
y_real = torch.tensor(results["Test Data"][1])

v = torch.mean((y_pred - y_real) ** 2)
sample_size = len(y_real)
MSE = torch.mean((y_pred - y_real)**2)
loss = -BIC_np_calc_loss(y_pred, y_real, torch.std(y_real), v, len(c), 16)
r_squared = 1 - torch.sum((y_pred - y_real)**2)/torch.sum((y_real - torch.mean(y_real))**2)
reward = NMSE_reward_func(y_pred.numpy(), y_real.numpy(), y_real.std())

print("Test Data")
print(f"R2: {r_squared}")
print(f"Reward: {reward}")
print(f"MSE: {MSE}")
print(f"NMSE: {MSE/(y_real**2).mean()}")
print(f"Loss: {loss}")

In [ ]:
# # x = torch.cat([torch.linspace(1, 5, 100).unsqueeze(0), torch.ones(100).unsqueeze(0)*5, torch.ones(100).unsqueeze(0)*5], dim=0)
# # x = torch.linspace(0, 4, 100).unsqueeze(0)
# # x = eval(results["Ground Truth"]["Dataset"]).unsqueeze(0)
# c = torch.tensor(results["Best Function"]["Constants"])
# y_pred = eval(results["Best Function"]["Equation"])
# # c = torch.tensor(results["Ground Truth"]["c"])
# y_real = eval(results["Ground Truth"]["Function"])

# # Plotting
# plt.figure(figsize=(8, 6))
# plt.plot(x[0], y_real, label="Ground Truth", color='blue')
# plt.plot(x[0], y_pred, label="Best Function", linestyle='dashed', color='red')

# plt.title("Comparison between Best Function and Ground Truth")
# plt.scatter(results["Run Info"]["X"], results["Run Info"]["Y"], label="Observed Data")
# plt.xlabel("x")
# plt.ylabel("y")
# plt.legend()
# plt.grid(True)
# plt.show() 

In [ ]:
# Plotting
plt.figure(figsize=(8, 6))
plt.title("Comparison between Best Function and Ground Truth")
plt.scatter(y_pred, y_real, label="Observed Data", s=20)
x = np.linspace(min(y_real), max(y_real), 100)
X = np.linspace(min(y_real), max(y_real), 100)
plt.plot(X, X, color="black")
color = 'tab:red'
# plt.fill_between(X, X - np.sqrt(results["Best Function"]["Noise"]),
#                  X +  np.sqrt(results["Best Function"]["Noise"]), color=color, alpha=0.2)
plt.xlabel("Modeled Y")
plt.ylabel("Actaul Y")
plt.legend()
plt.grid(True)
plt.show() 

In [ ]:
window_size = math.ceil(len(results['Training Cycle'][0]["Iteration Info"]["Loss"])/50)

In [ ]:
for i, iter in enumerate(results['Training Cycle'][0]["Iteration Info"]["Rewards"]):
    results['Training Cycle'][0]["Iteration Info"]["Rewards"][i] = [j for j in iter if j > -1E32]
results['Training Cycle'][0]["Iteration Info"]["Rewards"] = [np.array(r)[~np.isnan(r)] for r in results['Training Cycle'][0]["Iteration Info"]["Rewards"]]
results['Training Cycle'][0]["Iteration Info"]["Rewards"] = [np.array(r)[np.isfinite(r)] for r in results['Training Cycle'][0]["Iteration Info"]["Rewards"]]

In [ ]:
fig, ax = plt.subplots(1,1)
ax.plot(smooth_data(results['Training Cycle'][0]["Iteration Info"]["New Equations"], 1))
plt.show()

In [ ]:
fig, ax = plt.subplots(1,1)
tmp = [-0.005 * temp.to(torch.device('cpu')) for temp in results['Training Cycle'][0]["Iteration Info"]["Full Entropy"]]

smoothed_loss = smooth_data(results['Training Cycle'][0]["Iteration Info"]["Loss"], 1)
ax.plot(range(len(smoothed_loss)), smoothed_loss, label="Total Loss")
ax.plot(range(len(smoothed_loss)), smooth_data(results['Training Cycle'][0]["Iteration Info"]["Policy Loss"], 1), label="Policy Loss")
ax.plot(range(len(smoothed_loss)), smooth_data(results['Training Cycle'][0]["Iteration Info"]["Entropy Loss"], 1), label="Entropy Loss")
ax.plot(range(len(smoothed_loss)), smooth_data(results['Training Cycle'][0]["Iteration Info"]["KL Loss"], 1), label="KL Loss")


# Adding labels and title
ax.set_xlabel("Iteration")
ax.set_ylabel("Smoothed Loss")
ax.set_title("Different Losses")
# ax.set_xlim(0, 20)

ax.legend(title="Losses", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1)
node_means = [i.float().mean().to(torch.device("cpu")) for i in results['Training Cycle'][0]["Iteration Info"]["Node Counts"]]
node_median = [i.median().to(torch.device("cpu")) for i in results['Training Cycle'][0]["Iteration Info"]["Node Counts"]]
node_mode = [torch.mode(i.to(torch.device("cpu")))[0] for i in results['Training Cycle'][0]["Iteration Info"]["Node Counts"]]
node_stds =  [i.float().std().to(torch.device("cpu")) for i in results['Training Cycle'][0]["Iteration Info"]["Node Counts"]]

# Plotting the Best Reward with smoothing
smoothed_node_means = smooth_data(node_means, window_size)
smoothed_node_median = smooth_data(node_median, window_size)
smoothed_node_mode = smooth_data(node_mode, window_size)
smoothed_node_stds = smooth_data(node_stds, window_size)

x = range(len(smoothed_node_means))
color = 'tab:red'
ax.plot(x, smoothed_node_means, label="Node Means")
ax.plot(x, smoothed_node_median, label="Node Medians")
ax.plot(x, smoothed_node_mode, label="Node Modes")
ax.fill_between(x, np.array(smoothed_node_means) - np.array(smoothed_node_stds),
                 np.array(smoothed_node_means) + np.array(smoothed_node_stds), color=color, alpha=0.2)

# Adding labels and title
ax.set_xlabel("Iteration")
ax.set_ylabel("Node Counts")
ax.set_title("Smoothed Average Node Numbers per Expression Tree")

# Adding legend
ax.legend()

# Show the plot
plt.show()

In [ ]:
# Assuming 'results' is your data dictionary
node_nums = [i.to(torch.device("cpu")).numpy() for i in results['Training Cycle'][0]["Iteration Info"]["Node Counts"]]
num_epochs = len(node_nums)

# Create a continuous distribution plot with lines for each epoch
plt.figure(figsize=(12, 8))

# Define a single color from the 'viridis' colormap
base_color = mpl.colormaps["Blues"]
    
for epoch_number, node_num_epoch in enumerate(node_nums):
    node_num_epoch = sorted(node_num_epoch, reverse=True)
    sns.kdeplot(node_num_epoch, color=base_color(epoch_number/num_epochs), fill=False, label='', alpha=0.5, warn_singular=False)
plt.xlabel("Node Number")
plt.ylabel("Density")
# plt.ylim(0, 15)
plt.title("Continuous Distribution Plot for Expression Size Across Epochs")

# Create a continuous legend using a color bar outside the plot
cax = plt.axes([0.92, 0.1, 0.02, 0.8])  # Adjust the position and size of the color bar
cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=base_color), cax=cax)

cbar.set_label('Epoch Number')

plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(14, 7)

# Plotting the Best Reward with smoothing
window_size = 1
smoothed_best_reward = smooth_data(results['Training Cycle'][0]["Iteration Info"]["Best Reward"], window_size)
x = range(len(smoothed_best_reward))
# smoothed_best_reward = 1 / (1 + np.exp(-np.array(smoothed_best_reward)/500))
ax[0].plot(x, smoothed_best_reward, label="Best")

# Plotting the Median Reward with smoothing
smoothed_median_reward = smooth_data(results['Training Cycle'][0]["Iteration Info"]["Median Reward"], window_size)
# smoothed_median_reward = 1 / (1 + np.exp(-np.array(smoothed_median_reward)/500))
ax[0].plot(x, smoothed_median_reward, label=r'Median Top $\alpha\%$')

# Plotting the Baseline Reward with smoothing
smoothed_baseline_reward = smooth_data(results['Training Cycle'][0]["Iteration Info"]["Baseline Reward"], window_size)
# smoothed_baseline_reward = 1 / (1 + np.exp(-np.array(smoothed_baseline_reward)/500))
ax[0].plot(x, smoothed_baseline_reward, label=r'$R_\alpha$')

smoothed_min_reward = smooth_data([np.mean(np.array(i)) for i in results['Training Cycle'][0]["Iteration Info"]["Rewards"]], window_size)
# smoothed_min_reward = 1 / (1 + np.exp(-np.array(smoothed_min_reward)/500))
ax[0].plot(x, smoothed_min_reward, label="Median")

# Adding labels and title
ax[0].set_xlabel("Epoch", fontsize="14")
ax[0].set_ylabel("Reward", fontsize="14")
ax[0].set_title("(a)", fontsize="16", y=-0.25)
# ax.set_xlim(0, 50)
# ax[0].set_ylim(-700000, -600000)
# ax[0].set_ylim(-10000, -3000)
# Adding legend

ax[0].legend(loc="lower left", fontsize="10")

expressions_pruned = []
for i in range(len(results['Training Cycle'][0]["Iteration Info"]["Best Reward"])):
    current_rewards = results['Training Cycle'][0]["Iteration Info"]["Rewards"][i]
    epsilon = results['Training Cycle'][0]["Iteration Info"]["Baseline Reward"][i]
    current_rewards = [entry for entry in current_rewards if entry > epsilon]
    percent = 100 * (np.abs(current_rewards-epsilon) < 0.001).mean()
    expressions_pruned.append(percent)
smoothed_percent = smooth_data(expressions_pruned, window_size)

ax[1].plot(x, smoothed_percent)
ax[1].set_xlabel("Epoch", fontsize="14")
ax[1].set_ylabel("Percentage of Top $\\alpha$ % Expressions Pruned", fontsize="14")
ax[1].set_title("(b)", fontsize="16", y=-0.25)
# ax[1].set_ylim(-5, 40)

plt.show()
# fig.savefig("NMSE_unbiased_seed_314.pdf", dpi=500, bbox_inches='tight', pad_inches=0.1)

In [ ]:
fig, ax = plt.subplots(1, 1)
fig.set_size_inches(7, 7)


cum_best = [-np.inf]
for i, r in enumerate(results['Training Cycle'][0]["Iteration Info"]["Rewards"]):
    if max(r) > cum_best[-1]:
        cum_best.append(max(r))
    else:
        cum_best.append(cum_best[-1])

cum_epsilon_m = [-np.inf]
for i, r in enumerate(results['Training Cycle'][0]["Iteration Info"]["Median Reward"]):
    if r > cum_epsilon_m[-1]:
        cum_epsilon_m.append(r)
    else:
        cum_epsilon_m.append(cum_epsilon_m[-1])
        
cum_epsilon = [-np.inf]
for i, r in enumerate(results['Training Cycle'][0]["Iteration Info"]["Baseline Reward"]):
    if r > cum_epsilon[-1]:
        cum_epsilon.append(r)
    else:
        cum_epsilon.append(cum_epsilon[-1])
        
cum_median = [-np.inf]
for i, r in enumerate(results['Training Cycle'][0]["Iteration Info"]["Rewards"]):
    if np.median(r) > cum_median[-1]:
        cum_median.append(np.median(r))
    else:
        cum_median.append(cum_median[-1])
        
# Plotting the Best Reward with smoothing
window_size = 1
smoothed_best_reward = smooth_data(cum_best, window_size)
x = range(len(smoothed_best_reward))
# smoothed_best_reward = 1 / (1 + np.exp(-np.array(smoothed_best_reward)/500))

ax.plot(x, smoothed_best_reward, label="Best")

# Plotting the Epsilon Reward with smoothing
ax.plot(x, smooth_data(cum_epsilon_m, window_size), label=r'Median Top $\alpha\%$')

# Plotting the Epsilon Reward with smoothing
ax.plot(x, smooth_data(cum_epsilon, window_size), label=r'$R_\alpha$')

# Plotting the Median Reward with smoothing
ax.plot(x, smooth_data(cum_median, window_size), label="Median")


# Adding labels and title
ax.set_xlabel("Epoch", fontsize="14")
ax.set_ylabel("NMSE Reward", fontsize="14")
ax.set_title("Cummlative Performance", fontsize="16") # Cumulative Performance by BIC Loss
# ax.set_xlim(0,200)
# ax.set_ylim(79000, 82000)
# Adding legend
ax.legend( loc="center right")

# Show the plot
plt.show()
# fig.savefig("Mutations.png", dpi=500, bbox_inches='tight', pad_inches=0.3)

In [ ]:
# Assuming 'results' is your data dictionary
np.seterr(over='ignore')
rewards_per_epoch = results['Training Cycle'][0]["Iteration Info"]["Rewards"]
num_epochs = len(rewards_per_epoch)

# Create a continuous distribution plot with lines for each epoch
plt.figure(figsize=(12, 8))

# Define a single color from the 'viridis' colormap
base_color = mpl.colormaps["Blues"]
    
for epoch_number, rewards_for_single_epoch in enumerate(rewards_per_epoch):
    rewards_for_single_epoch = rewards_for_single_epoch[rewards_for_single_epoch != 0.0]
    rewards_for_single_epoch = sorted(rewards_for_single_epoch, reverse=True)
    # rewards_for_single_epoch = 1 / (1 + np.exp(-np.array(rewards_for_single_epoch)/1E4))
    sns.kdeplot(rewards_for_single_epoch, color=base_color(epoch_number/num_epochs), fill=False, label='', alpha=0.5, warn_singular=False)
plt.xlabel("Rewards")
plt.ylabel("Density")
plt.title("Continuous Distribution Plot for Rewards Across Epochs")

# Create a continuous legend using a color bar outside the plot
cax = plt.axes([0.92, 0.1, 0.02, 0.8])  # Adjust the position and size of the color bar
cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=base_color), cax=cax)

cbar.set_label('Epoch Number')

plt.show()

In [ ]:
# Assuming 'results' is your data dictionary
rewards_per_epoch = results['Training Cycle'][0]["Iteration Info"]["Rewards"]
num_epochs = len(rewards_per_epoch)
window_size = 1  # Set your desired window size

# Create a continuous distribution plot with lines for each epoch
plt.figure(figsize=(12, 8))

# Define a single color from the 'Blues' colormap
base_color = mpl.colormaps["Blues"]

for epoch_number, rewards_for_single_epoch in enumerate(rewards_per_epoch):
    rewards_for_single_epoch = sorted(rewards_for_single_epoch, reverse=True)[:50]
    # rewards_for_single_epoch = 1 / (1 + np.exp(-np.array(rewards_for_single_epoch)/1E4))
    sns.kdeplot(rewards_for_single_epoch, color=base_color(epoch_number / num_epochs), fill=False, label=f'Epoch {epoch_number}', alpha=0.5, warn_singular=False)

plt.xlabel("Rewards")
plt.ylabel("Density")
plt.ylim(0, 15)
plt.title("Continuous Distribution Plot for Top 50 Rewards Across Epochs")

# Create a continuous legend using a color bar outside the plot
cax = plt.axes([0.92, 0.1, 0.02, 0.8])  # Adjust the position and size of the color bar
cbar = plt.colorbar(plt.cm.ScalarMappable(cmap=base_color), cax=cax, ticks=[0, 1])
cbar.set_label('Epoch Number')

plt.show()

In [ ]:
# Assuming 'results' is your data dictionary
temp_rewards = []
for i, iter in enumerate(results['Training Cycle'][0]["Iteration Info"]["Rewards"]):
    temp_rewards.append([j for j in iter if j != 0])
rewards_entries = [len(entries) for entries in temp_rewards]
iterations = range(1, len(rewards_entries) + 1)

smoothed_entries = smooth_data(rewards_entries, window_size=1)

# Create a line plot with smoothing and without markers
plt.plot(iterations[len(iterations) - len(smoothed_entries):], smoothed_entries, linestyle='-', color='blue')

# Add labels and title
plt.xlabel("Iteration")
plt.ylabel("Smoothed Number of Entries in Rewards")
plt.title("Smoothed Number of Entries in Rewards Over Iterations")

# Show the plot
plt.show()

In [ ]:
# Assuming 'results' is your data dictionary
time_keys = ['Sample Time', 'Opt Time', 'Reward', 'Prediction']

# Example data (replace this with your actual data)
time_values = [sum(results['Training Cycle'][0]['Timings'][key]) for key in time_keys]
print(time_values)
time_values = [100 * v/sum(time_values) for v in time_values]

fig, ax = plt.subplots(figsize=(8, 8))

# Creating a pie chart with black lines separating sections
wedges, texts, autotexts = ax.pie(time_values, labels=None, autopct='', startangle=90, wedgeprops=dict(linewidth=2, edgecolor='black'))

# Creating a legend with labeled percents
legend_labels = [f"{key}: {time:.1f}%" for key, time in zip(time_keys, time_values)]
ax.legend(wedges, legend_labels, title="Time Components", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

# Adding title
ax.set_title("Overall Time Breakdown per Epoch")

# Show the plot
plt.show()


In [ ]:
# Assuming 'results' is your data dictionary
time_keys = results['Training Cycle'][0]['Timings']["Sample Time In-depth"][0].keys()
sample_timings = {}
for key in time_keys:
    sample_timings[key] = sum([results['Training Cycle'][0]['Timings']["Sample Time In-depth"][k][key] for k in range(len(results['Training Cycle'][0]['Timings']["Sample Time In-depth"]))])
sample_timings = (sample_timings.values())
sample_timings = [100 * v/sum(sample_timings) for v in sample_timings]

fig, ax = plt.subplots(figsize=(8, 8))

# Creating a pie chart with black lines separating sections
wedges, texts, autotexts = ax.pie(sample_timings, labels=None, autopct='', startangle=90, wedgeprops=dict(linewidth=2, edgecolor='black'))

# Creating a legend with labeled percents
legend_labels = [f"{key}: {time:.1f}%" for key, time in zip(time_keys, sample_timings)]
ax.legend(wedges, legend_labels, title="Time Components", loc="center left", bbox_to_anchor=(1, 0, 0.5, 1))

# Adding title
ax.set_title("Time Breakdown for Sampling Step")

# Show the plot
plt.show()
